# 02. Baseline Training

This notebook trains the baseline model for the Eye Disease classification project.

본 노트북은 안저 질환 분류 프로젝트의 기준 모델(Baseline)을 학습하는 코드입니다.

## Notebook Structure

1. Environment & Path Setup  
2. Reproducibility Setup  
3. Load Train / Validation CSV  
4. Optional Local Cache Setup  
5. Dataset & DataLoader  
6. Model Setup  
7. Training & Evaluation Functions  
8. Baseline Training  
9. Final Evaluation & Export  
10. Summary


## 1. Environment & Path Setup

Define project paths, output directories, and experiment configuration.

프로젝트 경로와 실험 결과 저장 위치를 설정합니다.


In [ ]:
from pathlib import Path
from datetime import datetime
import json
import os
import random
import platform
import sys
import shutil
import subprocess

import numpy as np
import pandas as pd

ROOT = Path("/content/drive/MyDrive/eye_disease_classifier")

# CSV files generated from 01_EDA_and_Data_Preparation.ipynb
SPLIT_DIR = ROOT / "splits"
CSV_TRAIN = SPLIT_DIR / "train.csv"
CSV_VAL = SPLIT_DIR / "val.csv"

# Optional local cache zip for faster training in Colab
DRV_ZIP_PATH = ROOT / "data" / "eye_disease_cached_256.zip"
ZIP_PATH = Path("/content/eye_disease_cached_256.zip")
CACHE_DIR = Path("/content/eye_disease_local_cached_256")

RUN_NAME = f"run_{datetime.now():%Y%m%d_%H%M%S}_baseline"
RUN_DIR = ROOT / "results" / RUN_NAME
CKPT_DIR = RUN_DIR / "checkpoints"
FIG_DIR = ROOT / "figures" / "results"

for d in [RUN_DIR, CKPT_DIR, FIG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("ROOT:", ROOT)
print("CSV_TRAIN:", CSV_TRAIN, CSV_TRAIN.exists())
print("CSV_VAL:", CSV_VAL, CSV_VAL.exists())
print("RUN_DIR:", RUN_DIR)
print("FIG_DIR:", FIG_DIR)


## 2. Reproducibility Setup

Set random seeds and record the training environment.

재현성을 위해 seed를 고정하고 실험 환경을 기록합니다.


In [ ]:
SEED = 42

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

    import torch
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(SEED)

def get_version(pkg_name):
    try:
        module = __import__(pkg_name)
        return getattr(module, "__version__", "unknown")
    except Exception:
        return "not installed"

env_info = {
    "python": sys.version,
    "platform": platform.platform(),
    "torch": get_version("torch"),
    "torchvision": get_version("torchvision"),
    "timm": get_version("timm"),
    "pandas": get_version("pandas"),
}

(RUN_DIR / "env_summary.json").write_text(json.dumps(env_info, indent=2))
env_info


## 3. Load Train / Validation CSV

Load train and validation split files created during EDA.

EDA 단계에서 생성한 train/validation CSV를 불러옵니다.


In [ ]:
assert CSV_TRAIN.exists(), f"Train CSV not found: {CSV_TRAIN}"
assert CSV_VAL.exists(), f"Validation CSV not found: {CSV_VAL}"

df_train = pd.read_csv(CSV_TRAIN)
df_val = pd.read_csv(CSV_VAL)

# Column compatibility
# Expected columns: path, class
# If your CSV uses image_path instead of path, convert it.
if "path" not in df_train.columns and "image_path" in df_train.columns:
    df_train = df_train.rename(columns={"image_path": "path"})
    df_val = df_val.rename(columns={"image_path": "path"})

required_cols = {"path", "class"}
assert required_cols.issubset(df_train.columns), f"Train CSV must contain {required_cols}"
assert required_cols.issubset(df_val.columns), f"Val CSV must contain {required_cols}"

classes = sorted(df_train["class"].unique().tolist())
class_to_idx = {cls_name: idx for idx, cls_name in enumerate(classes)}
idx_to_class = {idx: cls_name for cls_name, idx in class_to_idx.items()}

print("Number of classes:", len(classes))
print("Train size:", len(df_train))
print("Validation size:", len(df_val))
print("Classes:", classes)

with open(RUN_DIR / "class_to_idx.json", "w") as f:
    json.dump(class_to_idx, f, indent=2)


## 4. Optional Local Cache Setup

Training directly from Google Drive can be slow.  
If a cached ZIP file exists, this section copies and extracts the dataset to `/content` for faster data loading.

Google Drive에서 직접 이미지를 읽으면 느릴 수 있으므로, 캐시 ZIP이 있는 경우 `/content`로 복사하여 학습 속도를 개선합니다.


In [ ]:
USE_LOCAL_CACHE = True

def prepare_local_cache():
    if not DRV_ZIP_PATH.exists():
        print("[INFO] Cache ZIP not found. Training will use original image paths.")
        return False

    if not ZIP_PATH.exists():
        print("[INFO] Copying cache ZIP to /content ...")
        shutil.copy2(DRV_ZIP_PATH, ZIP_PATH)
    else:
        print("[SKIP] ZIP already exists:", ZIP_PATH)

    if not CACHE_DIR.exists() or not any(CACHE_DIR.iterdir()):
        print("[INFO] Unzipping dataset cache ...")
        subprocess.run(["unzip", "-q", str(ZIP_PATH), "-d", "/content/"], check=True)
    else:
        print("[SKIP] Cache directory already exists:", CACHE_DIR)

    return CACHE_DIR.exists()

cache_ready = prepare_local_cache() if USE_LOCAL_CACHE else False
print("cache_ready:", cache_ready)


In [ ]:
def map_to_cache_path(original_path):
    # Convert Drive image path to local cache path using class folder and filename.
    # This assumes each image filename is unique within its class folder.
    p = Path(original_path)
    class_name = p.parent.name
    filename = p.name
    cached_path = CACHE_DIR / class_name / filename

    if cached_path.exists():
        return str(cached_path)

    return str(original_path)

if cache_ready:
    df_train["path"] = df_train["path"].apply(map_to_cache_path)
    df_val["path"] = df_val["path"].apply(map_to_cache_path)

print("Sample train path:", df_train.iloc[0]["path"])
print("Path exists:", Path(df_train.iloc[0]["path"]).exists())


## 5. Dataset & DataLoader

Define PyTorch Dataset and DataLoader.

PyTorch Dataset과 DataLoader를 구성합니다.


In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

train_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Resize((256, 256)),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

val_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Resize((256, 256)),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

class FundusCSVDataset(Dataset):
    def __init__(self, dataframe, transform, class_to_idx):
        self.df = dataframe.reset_index(drop=True)
        self.transform = transform
        self.class_to_idx = class_to_idx

    def __len__(self):
        return len(self.df)

    def __getitem__(self, index):
        row = self.df.iloc[index]
        image = Image.open(row["path"]).convert("RGB")
        image = self.transform(image)
        label = self.class_to_idx[row["class"]]
        return image, label

train_dataset = FundusCSVDataset(df_train, train_transform, class_to_idx)
val_dataset = FundusCSVDataset(df_val, val_transform, class_to_idx)

BATCH_SIZE = 64
NUM_WORKERS = 2
PREFETCH_FACTOR = 2

loader_kwargs = {
    "batch_size": BATCH_SIZE,
    "num_workers": NUM_WORKERS,
    "pin_memory": device.type == "cuda",
}

if NUM_WORKERS > 0:
    loader_kwargs["persistent_workers"] = True
    loader_kwargs["prefetch_factor"] = PREFETCH_FACTOR

train_loader = DataLoader(train_dataset, shuffle=True, **loader_kwargs)
val_loader = DataLoader(val_dataset, shuffle=False, **loader_kwargs)

print("Train batches:", len(train_loader))
print("Validation batches:", len(val_loader))


## 6. Model Setup

Create the ConvNeXtV2 Tiny baseline model.

ConvNeXtV2 Tiny 기반 baseline 모델을 정의합니다.


In [ ]:
import timm
import torch.backends.cudnn as cudnn

cudnn.benchmark = True

MODEL_NAME = "convnextv2_tiny.fcmae_ft_in22k_in1k"
NUM_CLASSES = len(classes)

model = timm.create_model(
    MODEL_NAME,
    pretrained=True,
    num_classes=NUM_CLASSES
).to(device)

if device.type == "cuda":
    model = model.to(memory_format=torch.channels_last)

criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=0.05)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=20)

from torch import amp
scaler = amp.GradScaler(enabled=(device.type == "cuda"))

print("Model:", MODEL_NAME)
print("Number of classes:", NUM_CLASSES)


## 7. Training & Evaluation Functions

Define reusable training and evaluation functions.

학습과 평가 함수를 분리하여 가독성을 높입니다.


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt

def train_one_epoch(model, dataloader, optimizer, criterion, scaler, device):
    model.train()
    running_loss = 0.0

    for images, labels in dataloader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        if device.type == "cuda":
            images = images.to(memory_format=torch.channels_last)

        optimizer.zero_grad(set_to_none=True)

        with amp.autocast(device_type=device.type, enabled=(device.type == "cuda")):
            outputs = model(images)
            loss = criterion(outputs, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item() * images.size(0)

    return running_loss / len(dataloader.dataset)


def evaluate(model, dataloader, criterion, device):
    model.eval()

    running_loss = 0.0
    y_true = []
    y_pred = []

    with torch.inference_mode():
        for images, labels in dataloader:
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            if device.type == "cuda":
                images = images.to(memory_format=torch.channels_last)

            with amp.autocast(device_type=device.type, enabled=(device.type == "cuda")):
                outputs = model(images)
                loss = criterion(outputs, labels)

            preds = outputs.argmax(dim=1)

            running_loss += loss.item() * images.size(0)
            y_true.extend(labels.cpu().numpy())
            y_pred.extend(preds.cpu().numpy())

    val_loss = running_loss / len(dataloader.dataset)
    return val_loss, np.array(y_true), np.array(y_pred)


def save_confusion_matrix(y_true, y_pred, class_names, save_path):
    cm = confusion_matrix(y_true, y_pred, labels=list(range(len(class_names))))

    plt.figure(figsize=(8, 8))
    plt.imshow(cm, interpolation="nearest")
    plt.title("Confusion Matrix")
    plt.colorbar()
    plt.xlabel("Predicted label")
    plt.ylabel("True label")
    plt.tight_layout()
    plt.savefig(save_path, dpi=200, bbox_inches="tight")
    plt.close()

    return cm


## 8. Baseline Training

Train the baseline model and save the best checkpoint based on Macro F1-score.

Macro F1-score를 기준으로 가장 좋은 모델을 저장합니다.


In [ ]:
EPOCHS = 10
BEST_F1 = -1.0
history = []

print(f"Start baseline training for {EPOCHS} epochs")

for epoch in range(1, EPOCHS + 1):
    train_loss = train_one_epoch(
        model=model,
        dataloader=train_loader,
        optimizer=optimizer,
        criterion=criterion,
        scaler=scaler,
        device=device
    )

    scheduler.step()

    val_loss, y_true, y_pred = evaluate(
        model=model,
        dataloader=val_loader,
        criterion=criterion,
        device=device
    )

    report = classification_report(
        y_true,
        y_pred,
        target_names=classes,
        output_dict=True,
        zero_division=0
    )

    macro_f1 = report["macro avg"]["f1-score"]
    top1_acc = float((y_true == y_pred).mean())

    row = {
        "epoch": epoch,
        "train_loss": train_loss,
        "val_loss": val_loss,
        "top1_acc": top1_acc,
        "macro_precision": report["macro avg"]["precision"],
        "macro_recall": report["macro avg"]["recall"],
        "macro_f1": macro_f1,
        "lr": optimizer.param_groups[0]["lr"],
    }
    history.append(row)

    if macro_f1 > BEST_F1:
        BEST_F1 = macro_f1
        torch.save(model.state_dict(), CKPT_DIR / "best.pt")

    print(
        f"[Epoch {epoch:02d}] "
        f"train_loss={train_loss:.4f} "
        f"val_loss={val_loss:.4f} "
        f"acc={top1_acc:.4f} "
        f"macro_f1={macro_f1:.4f} "
        f"best={BEST_F1:.4f}"
    )

history_df = pd.DataFrame(history)
history_df.to_csv(RUN_DIR / "training_history.csv", index=False)

print("Best Macro F1:", BEST_F1)
print("Saved best checkpoint:", CKPT_DIR / "best.pt")


## 9. Final Evaluation & Export

Save final metrics, classification report, and confusion matrix.

최종 평가 결과와 리포트, confusion matrix를 저장합니다.


In [ ]:
# Reload best checkpoint before final evaluation
best_path = CKPT_DIR / "best.pt"
model.load_state_dict(torch.load(best_path, map_location=device))

val_loss, y_true, y_pred = evaluate(
    model=model,
    dataloader=val_loader,
    criterion=criterion,
    device=device
)

report = classification_report(
    y_true,
    y_pred,
    target_names=classes,
    output_dict=True,
    zero_division=0
)

report_df = pd.DataFrame(report).transpose()
report_path = RUN_DIR / "classification_report_baseline.csv"
report_df.to_csv(report_path)

metrics_summary = pd.DataFrame([{
    "val_loss": val_loss,
    "top1_acc": float((y_true == y_pred).mean()),
    "macro_precision": report["macro avg"]["precision"],
    "macro_recall": report["macro avg"]["recall"],
    "macro_f1": report["macro avg"]["f1-score"],
}])

metrics_path = RUN_DIR / "metrics_summary_baseline.csv"
metrics_summary.to_csv(metrics_path, index=False)

cm_path = FIG_DIR / "confusion_matrix_baseline.png"
cm = save_confusion_matrix(y_true, y_pred, classes, cm_path)

print("Saved report:", report_path)
print("Saved metrics:", metrics_path)
print("Saved confusion matrix:", cm_path)
display(metrics_summary)


## 10. Summary

This notebook trained a baseline ConvNeXtV2 Tiny model using standard CrossEntropyLoss.

본 노트북에서는 ConvNeXtV2 Tiny 기반 baseline 모델을 학습했습니다.

Key outputs:
- `best.pt`
- `training_history.csv`
- `classification_report_baseline.csv`
- `metrics_summary_baseline.csv`
- `confusion_matrix_baseline.png`

This baseline result is used as the reference point for later imbalance handling experiments.
